# Reinforcement learning on a stochastic Petri net

A net written with `zrth.petri` has two transitions that a policy controls. The policy is
a small `torch` network, trained by a policy-gradient algorithm, REINFORCE, on runs of
the net stepped through `zrth.simulate`. The objective is a time average over the run, and
a temporal property is checked on the runs at the end. The module is the environment:
nothing stands between the net and the learner.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch
import torch.nn as nn

from zrth import Module as compose
from zrth.petri import Arc, Place, Transition
from zrth.simulate import Simulation, simulate

## The system: one server, two queues

Customers arrive into queue A at rate $\lambda_A$ and into queue B at rate $\lambda_B$,
and one server serves either queue at rate $\mu$. Waiting costs $c_A$ per customer and unit
of time in A, $c_B$ in B. The server has to choose which queue to serve, and that choice is
the policy's.

```
arrive_a             queue_a               serve_a
  [λ_A] ------------> ( A ) --------------> [ μ ]
          arrival_a           departure_a     ^
                                              :
                                              : control_a
                                              :
                                       +---------------+
                                       |    policy     |
                                       | reads n_a n_b |
                                       +---------------+
                                              :
                                              : control_b
                                              :
          arrival_b           departure_b     v
  [λ_B] ------------> ( B ) --------------> [ μ ]
arrive_b             queue_b               serve_b
```

Every transition owns a clock armed with an exponential delay at its rate; a run flows
until the first clock expires, that transition fires, and so on, one round per expiry.
The two arrivals are plain transitions producing into their queue. `serve_a` and `serve_b`
consume from their queue and each has an enabling arc with no place behind it, a control
arc, dotted in the picture: nothing in the net writes its `enabled`, so it is an input of
the system, and whoever drives it decides whether that transition may fire.
The single server is the rule that exactly one control arc is enabled at a time.

In [ ]:
LAM_A, LAM_B, MU = 0.4, 0.4, 1.0
COST_A, COST_B = 1.0, 3.0

arrival_a, arrival_b = Arc("production"), Arc("production")
departure_a, departure_b = Arc("consumption"), Arc("consumption")
control_a, control_b = Arc("enabling"), Arc("enabling")

arrive_a = Transition(rate=LAM_A, outputs=[arrival_a])
arrive_b = Transition(rate=LAM_B, outputs=[arrival_b])
serve_a = Transition(rate=MU, inputs=[departure_a, control_a])
serve_b = Transition(rate=MU, inputs=[departure_b, control_b])
queue_a = Place(initial=0, inputs=[arrival_a], outputs=[departure_a])
queue_b = Place(initial=0, inputs=[arrival_b], outputs=[departure_b])

system = compose(arrive_a, arrive_b, serve_a, serve_b, queue_a, queue_b)
names = {control_a.enabled: "control_a", control_b.enabled: "control_b"}
print("inputs:", [names.get(v, "t") for v in system.extl])

inputs: ['t', 'control_a', 'control_b']


## Policies by hand

A policy is a function of the time and the state that returns the inputs of the next step;
`simulate` calls it before every step. Three hand-written ones to compare with later:

- the **$c\mu$ rule** of queueing theory: serve the queue with the largest $c \cdot \mu$,
  the cost removed per unit of service time, as long as it is non-empty. Here the service
  rates are equal, so it serves B whenever B is non-empty, else A;
- the **reversed priority**, A first;
- a **coin flip** at every decision.

In [ ]:
def choose(queue: str):
    return {control_a.enabled: queue == "A", control_b.enabled: queue == "B"}


def c_mu_rule(time, state):
    return choose("B" if state[queue_b.count] > 0 else "A")


def reversed_priority(time, state):
    return choose("A" if state[queue_a.count] > 0 else "B")


coin_rng = random.Random(0)


def coin_flip(time, state):
    return choose("AB"[coin_rng.random() < 0.5])


def plot_queues(trace, title):
    plt.figure(figsize=(9, 2.5))
    plt.step(trace.times, trace.series(queue_a.count), where="post", label="queue A")
    plt.step(trace.times, trace.series(queue_b.count), where="post", label="queue B")
    plt.xlabel("time"), plt.ylabel("customers"), plt.title(title), plt.legend(loc="upper right")
    plt.show()


plot_queues(simulate(system, horizon=100.0, seed=0, policy=c_mu_rule), "the cμ rule")

## The objective: a time average

Queue A grows while B is served, and the $c\mu$ rule is still the best choice, because the
objective is the cost of a run as a whole. With both service rates equal, the total number
waiting evolves the same whichever queue is served; what the choice decides is who waits,
and a customer waiting in B costs three times one in A. The cost of a run of length $T$ is
the time average of the holding cost,

$$J = \frac{1}{T} \int_0^T \big(c_A\, n_A(t) + c_B\, n_B(t)\big)\, dt :$$

the holding cost at each instant, integrated over the run and divided by its length, so
that $J$ is the cost per unit of time. The marking is constant between rounds, so the
integral is a sum over the trace: each state's cost times how long that state lasted. The
expectation over runs is an average over seeds.

In [ ]:
def cost(state):
    return COST_A * state[queue_a.count] + COST_B * state[queue_b.count]


def time_average(trace, f):
    total = 0.0
    for (t0, state), t1 in zip(trace, trace.times[1:]):
        total += f(state) * (t1 - t0)
    return total / trace.times[-1]


def mean_cost(policy, *, horizon, seeds):
    return sum(time_average(simulate(system, horizon=horizon, seed=s, policy=policy), cost) for s in seeds) / len(seeds)


EVAL_SEEDS = range(2000, 2020)
for name, policy in [("cμ rule", c_mu_rule), ("reversed priority", reversed_priority), ("coin flip", coin_flip)]:
    print(f"{name:18s} J = {mean_cost(policy, horizon=100.0, seeds=EVAL_SEEDS):.2f}")

cμ rule            J = 4.84


reversed priority  J = 8.11


coin flip          J = 11.54


## The policy network

The policy sees the marking, the two counts, and nothing else: the clocks are private to
the transitions and the events are not shown to it. Nothing is lost, since the clocks are
exponential and so memoryless: the marking is all the past tells about the future. The
network gives one score per queue, and the action is which queue to serve, sampled from
the softmax of the two scores.

In [ ]:
class PolicyNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 16)
        self.fc2 = nn.Linear(16, 2)

    def forward(self, s):
        return self.fc2(torch.tanh(self.fc1(s)))


def observe(state):
    return torch.tensor([state[queue_a.count], state[queue_b.count]], dtype=torch.float32)

## Training: REINFORCE

REINFORCE is the simplest policy-gradient algorithm: run the current policy, then make
the actions that were followed by a good outcome more probable and the others less. An
episode is a run of the system of length `HORIZON`, stepped by hand: read the marking from
the last snapshot, sample an action, give `step` the inputs, and take as reward minus the
holding cost of that marking times how long the step lasted, so that an episode's return
is $-T J$. A step ends at every expiry, so the policy decides at every event of the net.

The outcome credited to the action at step $t$ is the return-to-go $G_t$, the rewards from
$t$ on, discounted at a rate per unit of time so that a decision answers for the near
future rather than for the whole run. The loss over a batch of episodes is
$-\sum_t \log \pi(a_t \mid s_t)\, A_t$, where $A_t$ is $G_t$ standardised over the batch,
so that better-than-average outcomes push their actions up and worse ones push down. One
iteration is one batch of episodes and one gradient step.

In [ ]:
HORIZON, BATCH, DISCOUNT = 30.0, 8, 0.2


def episode(policy, seed):
    # one run of the system under the sampling policy: the log-probability of every
    # action taken, and the reward and duration of every step
    sim = Simulation(system, seed=seed)
    logps, rewards, lengths = [], [], []
    while sim.now < HORIZON:  # sim.now is the current time of the run
        state = sim.trace.states[-1]
        dist = torch.distributions.Categorical(logits=policy(observe(state)))
        action = dist.sample()
        logps.append(dist.log_prob(action))
        before = sim.now
        sim.step(choose("AB"[action.item()]), until=HORIZON)
        lengths.append(sim.now - before)
        rewards.append(-cost(state) * lengths[-1])
    return logps, rewards, lengths


def returns_to_go(rewards, lengths):
    # G_t = r_t + e^(-discount * dt) G_{t+1}: built from the last step backwards
    G, returns = 0.0, []
    for r, dt in zip(reversed(rewards), reversed(lengths)):
        G = r + math.exp(-DISCOUNT * dt) * G
        returns.append(G)
    return list(reversed(returns))


def train(policy, *, iterations, lr=0.02):
    opt = torch.optim.Adam(policy.parameters(), lr=lr)
    history = []  # the mean J of each batch
    for iteration in range(iterations):
        logps, advantages, costs = [], [], []
        for b in range(BATCH):
            # a new run per episode, from a fixed sequence of seeds
            lp, rewards, lengths = episode(policy, seed=iteration * BATCH + b)
            logps += lp
            advantages += returns_to_go(rewards, lengths)
            costs.append(-sum(rewards) / HORIZON)
        advantages = torch.tensor(advantages)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-6)
        loss = -(torch.stack(logps) * advantages).sum() / BATCH
        opt.zero_grad()
        loss.backward()
        opt.step()
        history.append(sum(costs) / BATCH)
        if iteration % 10 == 0 or iteration == iterations - 1:
            print(f"iteration {iteration:2d}: J = {history[-1]:.2f}")
    return history

In [ ]:
torch.manual_seed(0)
policy = PolicyNN()
history = train(policy, iterations=40)

iteration  0: J = 6.72


iteration 10: J = 5.75


iteration 20: J = 5.61


iteration 30: J = 2.84


iteration 39: J = 3.30


In [ ]:
reference = mean_cost(c_mu_rule, horizon=HORIZON, seeds=range(1000, 1020))
plt.figure(figsize=(9, 2.5))
plt.plot(history, label="training batches")
plt.axhline(reference, color="gray", linestyle="--", label="cμ rule")
plt.xlabel("iteration"), plt.ylabel("J"), plt.legend(loc="upper right")
plt.show()

## The learned policy

Training samples the action; the trained policy takes the queue with the larger score, and
wrapped in a function it is a policy like the hand-written ones. It is compared with them
over the evaluation seeds, which no training run used, on two numbers:

- $J$, the time-averaged holding cost, as above;
- $P(\Phi)$, the fraction of runs in which the property $\Phi = G_{[0,T]}\,(n_A + n_B \le
  C)$ holds: the total number waiting never exceeds the capacity $C$ during the run. That
  is the shape a control problem's constraint takes, a probability of a temporal property.
  The marking is piecewise constant, so $\Phi$ is a check on the snapshots, and the
  fraction over the seeds estimates its probability.

In [ ]:
CAPACITY = 8


def learned(time, state):
    return choose("AB"[policy(observe(state)).argmax().item()])


def holds(trace):
    return all(state[queue_a.count] + state[queue_b.count] <= CAPACITY for state in trace.states)


print(f"{'policy':18s} {'J':>6s} {'P(Φ)':>6s}")
for name, pol in [("learned", learned), ("cμ rule", c_mu_rule), ("reversed priority", reversed_priority), ("coin flip", coin_flip)]:
    traces = [simulate(system, horizon=100.0, seed=s, policy=pol) for s in EVAL_SEEDS]
    J = sum(time_average(tr, cost) for tr in traces) / len(traces)
    P = sum(holds(tr) for tr in traces) / len(traces)
    print(f"{name:18s} {J:6.2f} {P:6.2f}")

plot_queues(simulate(system, horizon=100.0, seed=0, policy=learned), "the learned policy, same seed as the cμ rule above")

policy                  J   P(Φ)


learned              4.84   0.55


cμ rule              4.84   0.55


reversed priority    8.11   0.45


coin flip           11.63   0.20


The decision table says what was learned: the queue the trained policy serves for each
marking, $n_A$ down the rows and $n_B$ across the columns. The $c\mu$ rule reads B in every
column but the first, A there.

In [ ]:
print("   " + " ".join(f"{nb:2d}" for nb in range(6)))
for na in range(6):
    row = ["AB"[policy(torch.tensor([na, nb], dtype=torch.float32)).argmax().item()] for nb in range(6)]
    print(f"{na:2d} " + " ".join(f"{a:>2s}" for a in row))

    0  1  2  3  4  5
 0  A  B  B  B  B  B
 1  A  B  B  B  B  B
 2  A  B  B  B  B  B
 3  A  B  B  B  B  B
 4  A  B  B  B  B  B
 5  A  B  B  B  B  B
